# Code for plotting results from cpc-llm pipeline

By Drew Prinster

The following code plots the results for the cpc-llm pipeline (fraction infeasible, average score per round, max score over rounds) directly from each round's `round_summary.json`, using the `ar_sampling.accepted_quality` field. Note that scores in `round_summary.json` are negated relative to the raw score files, so they are re-negated here. For round 0, there is no `round_summary.json` because the initial calibration and training data are genearted directly from the safe policy without any CPC calibration, so round 0 data are calculated from a shared file for each CPC alpha value. When averaging a quantity across trials, the average is weighted by that trial/round's `n_samples` (most rounds have 200, but some have fewer).

In [ ]:
## Set directory paths and setting_str corresponding to config. Note: parent_output_dir (from config) == {MOUNT_DIR}/{REPO_DIR_NAME}/{PARENT_DIR}

MOUNT_DIR = '/cv/data/ai4dd/data/prinstea'
REPO_DIR_NAME = 'conformal-policy-control/cpc_llm'

PARENT_DIR = 'parent_output_20260625_bestScoreSafe'
# 'parent_output_replication_20260730',

SETTING_STR = '20260629_200PostCPC_0.333calFrac_0.2oldSeed_1.5e-7lr_fft0.001_minBetaMix1e-80_CPCfix'
# 'cpc_llm_replication_20260730',

print(SETTING_STR)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

def plot_results_from_round_summary(setting_str,
                                     parent_dir,
                                     mount_dir,
                                     repo_dir_name='llome',
                                     init_iters=100,
                                     alphas=[0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0][::-1],
                                     max_steps=10,
                                     start_seed=0,
                                     last_seed=9,
                                     retrain_initial_sft_each_trial=False,
                                     plot_min_infeas=False,
                                     plot_titles=False,
                                     solid_err_regions=False,
                                     title=None,
                                     dpi=100,
                                     fig_w=24,
                                     fig_h=4,
                                     last_sft_round_idx=0,
                                     n_legend_cols=4):
    """Same three plots as plot_results, but sourced from each round's round_summary.json
    (ar_sampling.accepted_quality) instead of the raw accepted_*.jsonl result files.

    round_summary.json stores negated scores, so mean_score/min_score are re-negated here.
    There is no round_summary.json for the round-0 SFT-init directory (round-0 generations
    are shared across alpha levels), so round 0 is instead read from the same
    gens_likelihood_dpo* file that plot_results uses for that round. Cross-trial averages
    for fraction infeasible and mean score are both weighted by each trial/round's
    accepted_quality.n_samples (usually 200, but sometimes fewer). Rounds where every
    accepted sample was infeasible (score NaN/inf) contribute 0 for mean score and for the
    round's contribution to the running max score, matching plot_results' np.nan_to_num
    handling of the equivalent case; weighting by n_samples (rather than n_feasible) keeps
    such all-infeasible rounds included at full weight in the mean-score average, since an
    all-infeasible round is a bad outcome that should pull the average down, not be
    excluded from it.
    """

    BLUE = '#2166ac'           # Standard blue for constrained policy color
    TEAL = '#5ab4ac'           # Color for safe policy
    RED = '#d73027'            # Color for unconstrained policy
    colors = {0.3 : 'C2', 0.4 : 'C9', 0.5 : 'C2', 0.6 : BLUE, 0.7 : 'C4', 0.8 : 'C4', 0.9 : 'C6', 1.0 : RED}
    markers = {0.3 : 'o', 0.4 : 'o', 0.5 : 'o', 0.6 : 'o', 0.7 : 'o', 0.8 : 'o', 0.9 : 'o', 1.0 : 'X'}
    n_pi_steps = max_steps
    n_trials = last_seed - start_seed + 1

    if plot_min_infeas:
        fig, axes = plt.subplots(1, 4, gridspec_kw={'width_ratios': [1, 1, 1, 1]}, figsize=(fig_w, fig_h))
        (ax1, ax2, ax3, ax4) = axes
    else:
        fig, axes = plt.subplots(1, 3, gridspec_kw={'width_ratios': [1, 1, 1]}, figsize=(fig_w, fig_h))
        (ax1, ax3, ax4) = axes

    plt.subplots_adjust(wspace=0.4)

    title_size = 16
    axs_size = 16

    ax1.set_ylabel(r'Fraction Infeasible (Risk) [$\leftarrow$]', fontsize=axs_size)
    ax1.set_xlabel('Timestep of Policy Improvement', fontsize=axs_size)
    ax1.set_ylim([0, 1])

    if plot_min_infeas:
        if plot_titles:
            ax2.set_title(r'Min Average Infeasible Rate so far', fontsize=title_size)
        ax2.set_ylabel(r'Min Avg Feasible Rate [$\leftarrow$]', fontsize=axs_size)
        ax2.set_xlabel('Timestep of Policy Improvement', fontsize=axs_size)
        ax2.set_ylim([0, 1])

    ax3.set_xlabel('Timestep of Policy Improvement', fontsize=axs_size)
    ax3.set_ylabel(r'Avg. Score per Round [$\rightarrow$]', fontsize=axs_size)

    ax4.set_xlabel('Timestep of Policy Improvement', fontsize=axs_size)
    ax4.set_ylabel(r'Max Score Over Rounds [$\rightarrow$]', fontsize=axs_size)

    if plot_titles:
        ax1.set_title(r'Fraction Infeasible', fontsize=title_size)
        ax3.set_title('Average Ehrlich score \n (Among Feasible) for Current Round', fontsize=title_size)
        ax4.set_title('Maximum score over all rounds \n(Averaged over experiments)', fontsize=title_size)

    uncontrolled_alpha_idx = 0
    linestyles = ['-', '--', '-.', ':']

    for alpha in tqdm(alphas):
        alpha_str = str(alpha)

        # Per-trial, per-round arrays; -1 indicates "no data available".
        frac_feasible_arr = -np.ones((n_trials, n_pi_steps))
        n_samples_arr = np.zeros((n_trials, n_pi_steps))
        mean_score_arr = -np.ones((n_trials, n_pi_steps))
        max_score_arr = -np.ones((n_trials, n_pi_steps))

        for random_seed in range(start_seed, last_seed + 1):


            setting_str_with_seed = f'seed{random_seed}_{setting_str}'
            row = random_seed - start_seed
            running_max = None

            # Round 0's generations are shared across alpha levels and have no
            # round_summary.json, so compute this round directly from the raw
            # gens_likelihood_dpo* file, same as plot_results does.
            if retrain_initial_sft_each_trial:
                fp0 = f'{mount_dir}/{repo_dir_name}/{parent_dir}/smaller_pythia/sft_init_seed{random_seed}_r{last_sft_round_idx}/gens_likelihood_dpo_{setting_str_with_seed}_20sample_{init_iters}iter_temp1.0_10seqs.jsonl'
            else:
                fp0 = f'{mount_dir}/{repo_dir_name}/{parent_dir}/smaller_pythia/smaller_pythia_sft_init_r{last_sft_round_idx}/gens_likelihood_dpo_{setting_str_with_seed}_20sample_{init_iters}iter_temp1.0_10seqs.jsonl'

            if not os.path.isfile(fp0):
                # Matches plot_results: if round 0 is missing, the trial contributes no data.
                continue

            data0 = pd.read_json(fp0, orient="records", lines=True)
            infeasible0 = np.array(data0['score'].isna())
            n0 = len(data0)

            n_feasible0 = n0 - infeasible0.sum()
            frac_feasible_arr[row, 0] = n_feasible0 / n0
            n_samples_arr[row, 0] = n0

            mean0 = -data0[~infeasible0]['score'].mean()
            mean_score_arr[row, 0] = mean0 if np.isfinite(mean0) else 0.0

            max0 = (-data0['score']).max()
            running_max = max0 if np.isfinite(max0) else 0.0
            max_score_arr[row, 0] = running_max

            for i in range(1, n_pi_steps):
                fp = f'{mount_dir}/{repo_dir_name}/{parent_dir}/smaller_pythia/smaller_pythia_alpha{alpha}_dpo_{setting_str_with_seed}_r{i}/round_summary.json'

                if not os.path.isfile(fp):
                    break

                with open(fp) as f:
                    round_summary = json.load(f)

                aq = round_summary['ar_sampling']['accepted_quality']

                frac_feasible_arr[row, i] = aq['frac_feasible']
                n_samples_arr[row, i] = aq['n_samples']

                # A round where every accepted sample was infeasible has NaN
                # mean_score/min_score in round_summary.json; treat these as 0,
                # matching plot_results' np.nan_to_num handling of the same case.
                mean_score = -aq['mean_score']
                mean_score_arr[row, i] = mean_score if np.isfinite(mean_score) else 0.0

                round_max_score = -aq['min_score']
                if not np.isfinite(round_max_score):
                    round_max_score = 0.0
                running_max = round_max_score if running_max is None else max(running_max, round_max_score)
                max_score_arr[row, i] = running_max

        # Cross-trial aggregation. Fraction-feasible and mean-score are both averaged
        # across trials weighted by accepted_quality.n_samples; max-score is a plain
        # (unweighted) average.
        frac_feasible_mean = np.full(n_pi_steps, np.nan)
        frac_feasible_stderr = np.full(n_pi_steps, np.nan)
        scores_mean = np.full(n_pi_steps, np.nan)
        scores_stderr = np.full(n_pi_steps, np.nan)
        max_scores_mean = np.full(n_pi_steps, np.nan)
        max_scores_stderr = np.full(n_pi_steps, np.nan)

        for i in range(n_pi_steps):
            valid = frac_feasible_arr[:, i] >= 0
            n_valid = np.sum(valid)
            if n_valid == 0:
                continue

            w = n_samples_arr[valid, i]

            ff = frac_feasible_arr[valid, i]
            ff_mean = np.average(ff, weights=w)
            frac_feasible_mean[i] = ff_mean
            frac_feasible_stderr[i] = np.sqrt(np.average((ff - ff_mean) ** 2, weights=w)) / np.sqrt(n_valid)

            sc = mean_score_arr[valid, i]
            sc_mean = np.average(sc, weights=w)
            scores_mean[i] = sc_mean
            scores_stderr[i] = np.sqrt(np.average((sc - sc_mean) ** 2, weights=w)) / np.sqrt(n_valid)

            ms = max_score_arr[valid, i]
            max_scores_mean[i] = np.mean(ms)
            max_scores_stderr[i] = np.std(ms) / np.sqrt(n_valid)

        marker = markers[alpha]
        linewidth = 2
        markersize = 8
        capsize = 3

        if alpha == 1.0:
            ax1.axhline(y=alpha, label=fr'B=1.0', linestyle=':', color='gray')
        elif alpha < 1.0:
            ax1.axhline(y=alpha, label=fr'Target $\alpha$={alpha}', linestyle='--', color=colors[alpha])

        if alpha >= 1.0:
            label = rf'No CPC'
            linestyle = linestyles[int(uncontrolled_alpha_idx % len(linestyles))]
            uncontrolled_alpha_idx += 1
        else:
            label = rf'CPC, $\alpha$={round(alpha,1)}'
            linestyle = '-'

        valid_steps = ~np.isnan(frac_feasible_mean)
        xs = np.arange(n_pi_steps)[valid_steps]

        if solid_err_regions:
            ax1.fill_between(xs, (1 - frac_feasible_mean - frac_feasible_stderr)[valid_steps], (1 - frac_feasible_mean + frac_feasible_stderr)[valid_steps], alpha=0.2, color=colors[alpha])
            ax1.plot(xs, (1 - frac_feasible_mean)[valid_steps], label=label, linewidth=linewidth, marker=marker, ms=markersize, alpha=0.85, color=colors[alpha], ls=linestyle)

            ax3.fill_between(xs, (scores_mean - scores_stderr)[valid_steps], (scores_mean + scores_stderr)[valid_steps], alpha=0.2, color=colors[alpha])
            ax3.plot(xs, scores_mean[valid_steps], label=label, linewidth=linewidth, marker=marker, ms=markersize, alpha=0.85, color=colors[alpha], ls=linestyle)

            ax4.fill_between(xs, (max_scores_mean - max_scores_stderr)[valid_steps], (max_scores_mean + max_scores_stderr)[valid_steps], alpha=0.2, color=colors[alpha])
            ax4.plot(xs, max_scores_mean[valid_steps], label=label, linewidth=linewidth, marker=marker, ms=markersize, alpha=0.85, color=colors[alpha], ls=linestyle)
        else:
            ax1.errorbar(xs, (1 - frac_feasible_mean)[valid_steps], yerr=frac_feasible_stderr[valid_steps], capsize=capsize, linewidth=linewidth, marker=marker, ms=markersize, label=label, alpha=0.8, color=colors[alpha])
            ax3.errorbar(xs, scores_mean[valid_steps], yerr=scores_stderr[valid_steps], capsize=capsize, linewidth=linewidth, marker=marker, ms=markersize, label=label, alpha=0.8, color=colors[alpha])
            ax4.errorbar(xs, max_scores_mean[valid_steps], yerr=max_scores_stderr[valid_steps], capsize=capsize, marker=marker, ms=markersize, label=label, alpha=0.8, color=colors[alpha])

    for i, ax in enumerate(axes):
        ax.spines[['right', 'top']].set_visible(False)
        ax.tick_params(axis='both', labelsize=14)

    ## Reorder labels in legend
    handles, labels = ax1.get_legend_handles_labels()
    leg1 = ax1.legend(handles, labels, loc=[-0.25, 1.3], ncols=n_legend_cols, fontsize=14)

    leg1.set_in_layout(False)

    legend_extent = leg1.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())

    fig.savefig(f'legend_roundsummary_{setting_str}_solidErrs{solid_err_regions}_seeds{start_seed}-{last_seed}_{len(alphas)}alphas.pdf', bbox_inches=legend_extent, dpi=dpi)

    extent1 = ax1.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
    extent3 = ax3.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
    extent4 = ax4.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())

    # Save the figure, cropping to the extent of the current axis
    plt.savefig(f'Infeasibility_roundsummary_{setting_str}_solidErrs{solid_err_regions}_seeds{start_seed}-{last_seed}_{len(alphas)}alphas_w{fig_w}_h{fig_h}.pdf', bbox_inches=extent1.expanded(1.1, 1.1), dpi=dpi)
    plt.savefig(f'AvgScores_roundsummary_{setting_str}_solidErrs{solid_err_regions}_seeds{start_seed}-{last_seed}_{len(alphas)}alphas_w{fig_w}_h{fig_h}.pdf', bbox_inches=extent3.expanded(1.1, 1.1), dpi=dpi)
    plt.savefig(f'MaxScores_roundsummary_{setting_str}_solidErrs{solid_err_regions}_seeds{start_seed}-{last_seed}_{len(alphas)}alphas_w{fig_w}_h{fig_h}.pdf', bbox_inches=extent4.expanded(1.1, 1.1), dpi=dpi)

    if plot_min_infeas:
        extent2 = ax2.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
        plt.savefig(f'MinInfeasibility_roundsummary_{setting_str}_solidErrs{solid_err_regions}_seeds{start_seed}-{last_seed}_{len(alphas)}alphas_w{fig_w}_h{fig_h}.pdf', bbox_inches=extent2.expanded(1.05, 1.05))

    leg1.set_in_layout(True)
    plt.show()

In [ ]:
plot_results_from_round_summary(setting_str=SETTING_STR,
         parent_dir = PARENT_DIR,
         mount_dir = MOUNT_DIR,
         repo_dir_name = REPO_DIR_NAME,
         alphas=[0.4, 0.6, 0.8, 1.0][::-1],
         plot_min_infeas = False, 
         plot_titles=False, 
         start_seed=0, 
         last_seed=29,
         retrain_initial_sft_each_trial=True,
         solid_err_regions=True,
         fig_w=16,
         max_steps = 20,
         last_sft_round_idx=0,
         init_iters=25, 
         dpi=300,
         n_legend_cols=4
        )

## Sample efficiency plots (from `round_summary.json`)

These plots use the `ar_sampling` field of each round's `round_summary.json` (acceptance-rejection sampling only happens from round 1 onward, so there is no round-0 data point here, unlike the plots above). Three subplots, sharing the same x-axis as the plots above: (a) acceptance probability (`ar_sampling.acceptance_rate`), (b) number of valid proposal sequences needed per accepted sample (`ar_sampling.n_proposals_total / ar_sampling.n_accepted`), and (c) number of total sequence-generation calls needed per accepted sample (`ar_sampling.n_calls * seqs_per_call / ar_sampling.n_accepted`), where `seqs_per_call = iterative_generation.args.sample_size * iterative_generation.args.max_iterations * sampling_gen_batch_size` is read from `cpc_llm.yaml` (falling back to a hardcoded default if the config can't be found). All three are simple (unweighted) per-round averages across trials.

In [ ]:
import yaml


def _get_seqs_per_call(config_path, fallback):
    try:
        with open(config_path) as f:
            cfg = yaml.safe_load(f)
        ig_args = cfg['iterative_generation']['args']
        return ig_args['sample_size'] * ig_args['max_iterations'] * cfg['sampling_gen_batch_size']
    except (OSError, KeyError, TypeError) as e:
        print(f"Could not load seqs_per_call from {config_path} ({e}); using fallback {fallback}")
        return fallback


def plot_sample_efficiency_from_round_summary(setting_str,
                                                parent_dir,
                                                mount_dir,
                                                repo_dir_name='llome',
                                                alphas=[0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0][::-1],
                                                max_steps=10,
                                                start_seed=0,
                                                last_seed=9,
                                                plot_titles=False,
                                                solid_err_regions=False,
                                                dpi=100,
                                                fig_w=24,
                                                fig_h=4,
                                                n_legend_cols=4,
                                                config_path='/cv/home/prinstea/conformal-policy-control/cpc_llm/config/cpc_llm.yaml',
                                                seqs_per_call_fallback=8000):
    """Sample-efficiency plots sourced from each round's round_summary.json (ar_sampling).

    There is no ar_sampling for the round-0 SFT-init round (acceptance-rejection sampling
    only happens from round 1 onward), so these series start at round 1.

    seqs_per_call is iterative_generation.args.sample_size * .max_iterations *
    sampling_gen_batch_size from cpc_llm.yaml (i.e. the number of raw sequences generated
    per AR-sampling call); it is loaded from config_path, falling back to
    seqs_per_call_fallback if the config can't be read.
    """
    seqs_per_call = _get_seqs_per_call(config_path, seqs_per_call_fallback)

    BLUE = '#2166ac'
    TEAL = '#5ab4ac'
    RED = '#d73027'
    colors = {0.3 : 'C2', 0.4 : 'C9', 0.5 : 'C2', 0.6 : BLUE, 0.7 : 'C4', 0.8 : 'C4', 0.9 : 'C6', 1.0 : RED}
    markers = {0.3 : 'o', 0.4 : 'o', 0.5 : 'o', 0.6 : 'o', 0.7 : 'o', 0.8 : 'o', 0.9 : 'o', 1.0 : 'X'}
    n_pi_steps = max_steps
    n_trials = last_seed - start_seed + 1

    fig, axes = plt.subplots(1, 3, gridspec_kw={'width_ratios': [1, 1, 1]}, figsize=(fig_w, fig_h))
    (ax1, ax2, ax3) = axes
    plt.subplots_adjust(wspace=0.4)

    axs_size = 16
    title_size = 16

    ax1.set_xlabel('Timestep of Policy Improvement', fontsize=axs_size)
    ax1.set_ylabel(r'Acceptance Probability [$\rightarrow$]', fontsize=axs_size)

    ax2.set_xlabel('Timestep of Policy Improvement', fontsize=axs_size)
    ax2.set_ylabel(r'Num Valid Proposal Sequences per One Accepted Sample [$\leftarrow$]', fontsize=axs_size)

    ax3.set_xlabel('Timestep of Policy Improvement', fontsize=axs_size)
    ax3.set_ylabel(r'Num Total Sequence Generation Calls per One Accepted Sample [$\leftarrow$]', fontsize=axs_size)

    if plot_titles:
        ax1.set_title('Acceptance Rate', fontsize=title_size)
        ax2.set_title('Proposals per Accepted Sample', fontsize=title_size)
        ax3.set_title('Generation Calls per Accepted Sample', fontsize=title_size)

    uncontrolled_alpha_idx = 0
    linestyles = ['-', '--', '-.', ':']

    for alpha in tqdm(alphas):

        # Per-trial, per-round arrays; -1 indicates "no data available".
        accept_rate_arr = -np.ones((n_trials, n_pi_steps))
        proposals_per_accept_arr = -np.ones((n_trials, n_pi_steps))
        calls_per_accept_arr = -np.ones((n_trials, n_pi_steps))

        for random_seed in range(start_seed, last_seed + 1):

            setting_str_with_seed = f'seed{random_seed}_{setting_str}'
            row = random_seed - start_seed

            # ar_sampling only happens from round 1 onward; round 0 (SFT-init) has no
            # round_summary.json and no acceptance-rejection sampling.
            for i in range(1, n_pi_steps):
                fp = f'{mount_dir}/{repo_dir_name}/{parent_dir}/smaller_pythia/smaller_pythia_alpha{alpha}_dpo_{setting_str_with_seed}_r{i}/round_summary.json'

                if not os.path.isfile(fp):
                    break

                with open(fp) as f:
                    round_summary = json.load(f)

                ar = round_summary['ar_sampling']
                n_accepted = ar['n_accepted']
                if n_accepted <= 0:
                    break

                accept_rate_arr[row, i] = ar['acceptance_rate']
                proposals_per_accept_arr[row, i] = ar['n_proposals_total'] / n_accepted
                calls_per_accept_arr[row, i] = (ar['n_calls'] * seqs_per_call) / n_accepted

        accept_rate_mean = np.full(n_pi_steps, np.nan)
        accept_rate_stderr = np.full(n_pi_steps, np.nan)
        proposals_per_accept_mean = np.full(n_pi_steps, np.nan)
        proposals_per_accept_stderr = np.full(n_pi_steps, np.nan)
        calls_per_accept_mean = np.full(n_pi_steps, np.nan)
        calls_per_accept_stderr = np.full(n_pi_steps, np.nan)

        for i in range(n_pi_steps):
            valid = accept_rate_arr[:, i] >= 0
            n_valid = np.sum(valid)
            if n_valid == 0:
                continue

            accept_rate_mean[i] = np.mean(accept_rate_arr[valid, i])
            accept_rate_stderr[i] = np.std(accept_rate_arr[valid, i]) / np.sqrt(n_valid)

            proposals_per_accept_mean[i] = np.mean(proposals_per_accept_arr[valid, i])
            proposals_per_accept_stderr[i] = np.std(proposals_per_accept_arr[valid, i]) / np.sqrt(n_valid)

            calls_per_accept_mean[i] = np.mean(calls_per_accept_arr[valid, i])
            calls_per_accept_stderr[i] = np.std(calls_per_accept_arr[valid, i]) / np.sqrt(n_valid)

        marker = markers[alpha]
        linewidth = 2
        markersize = 8
        capsize = 3

        if alpha >= 1.0:
            label = rf'No CPC'
            linestyle = linestyles[int(uncontrolled_alpha_idx % len(linestyles))]
            uncontrolled_alpha_idx += 1
        else:
            label = rf'CPC, $\alpha$={round(alpha,1)}'
            linestyle = '-'

        valid_steps = ~np.isnan(accept_rate_mean)
        xs = np.arange(n_pi_steps)[valid_steps]

        if solid_err_regions:
            ax1.fill_between(xs, (accept_rate_mean - accept_rate_stderr)[valid_steps], (accept_rate_mean + accept_rate_stderr)[valid_steps], alpha=0.2, color=colors[alpha])
            ax1.plot(xs, accept_rate_mean[valid_steps], label=label, linewidth=linewidth, marker=marker, ms=markersize, alpha=0.85, color=colors[alpha], ls=linestyle)

            ax2.fill_between(xs, (proposals_per_accept_mean - proposals_per_accept_stderr)[valid_steps], (proposals_per_accept_mean + proposals_per_accept_stderr)[valid_steps], alpha=0.2, color=colors[alpha])
            ax2.plot(xs, proposals_per_accept_mean[valid_steps], label=label, linewidth=linewidth, marker=marker, ms=markersize, alpha=0.85, color=colors[alpha], ls=linestyle)

            ax3.fill_between(xs, (calls_per_accept_mean - calls_per_accept_stderr)[valid_steps], (calls_per_accept_mean + calls_per_accept_stderr)[valid_steps], alpha=0.2, color=colors[alpha])
            ax3.plot(xs, calls_per_accept_mean[valid_steps], label=label, linewidth=linewidth, marker=marker, ms=markersize, alpha=0.85, color=colors[alpha], ls=linestyle)
        else:
            ax1.errorbar(xs, accept_rate_mean[valid_steps], yerr=accept_rate_stderr[valid_steps], capsize=capsize, linewidth=linewidth, marker=marker, ms=markersize, label=label, alpha=0.8, color=colors[alpha])
            ax2.errorbar(xs, proposals_per_accept_mean[valid_steps], yerr=proposals_per_accept_stderr[valid_steps], capsize=capsize, linewidth=linewidth, marker=marker, ms=markersize, label=label, alpha=0.8, color=colors[alpha])
            ax3.errorbar(xs, calls_per_accept_mean[valid_steps], yerr=calls_per_accept_stderr[valid_steps], capsize=capsize, linewidth=linewidth, marker=marker, ms=markersize, label=label, alpha=0.8, color=colors[alpha])

    for ax in axes:
        ax.spines[['right', 'top']].set_visible(False)
        ax.tick_params(axis='both', labelsize=14)

    handles, labels = ax1.get_legend_handles_labels()
    leg1 = ax1.legend(handles, labels, loc=[-0.25, 1.3], ncols=n_legend_cols, fontsize=14)
    leg1.set_in_layout(False)

    legend_extent = leg1.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
    fig.savefig(f'legend_sampleEff_roundsummary_{setting_str}_solidErrs{solid_err_regions}_seeds{start_seed}-{last_seed}_{len(alphas)}alphas.pdf', bbox_inches=legend_extent, dpi=dpi)

    extent1 = ax1.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
    extent2 = ax2.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
    extent3 = ax3.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())

    plt.savefig(f'AcceptanceRate_roundsummary_{setting_str}_solidErrs{solid_err_regions}_seeds{start_seed}-{last_seed}_{len(alphas)}alphas_w{fig_w}_h{fig_h}.pdf', bbox_inches=extent1.expanded(1.1, 1.1), dpi=dpi)
    plt.savefig(f'ProposalsPerAccepted_roundsummary_{setting_str}_solidErrs{solid_err_regions}_seeds{start_seed}-{last_seed}_{len(alphas)}alphas_w{fig_w}_h{fig_h}.pdf', bbox_inches=extent2.expanded(1.1, 1.1), dpi=dpi)
    plt.savefig(f'CallsPerAccepted_roundsummary_{setting_str}_solidErrs{solid_err_regions}_seeds{start_seed}-{last_seed}_{len(alphas)}alphas_w{fig_w}_h{fig_h}.pdf', bbox_inches=extent3.expanded(1.1, 1.1), dpi=dpi)

    plt.show()

In [ ]:
plot_sample_efficiency_from_round_summary(setting_str=SETTING_STR,
         parent_dir = PARENT_DIR,
         mount_dir = MOUNT_DIR,
         repo_dir_name = REPO_DIR_NAME,
         alphas=[0.4, 0.6, 0.8, 1.0][::-1],
         plot_titles=False,
         start_seed=0,
         last_seed=29,
         solid_err_regions=True,
         fig_w=16,
         max_steps = 20,
         dpi=300,
         n_legend_cols=4
        )